# M3 Event Classifier (`m3_event_clf`) — DeBERTa-v3-base 4-Class + ONNX `fp32`

- **Task (`AR-08`):** 4-class classification (`actual_event`, `hypothetical`, `negated`, `not_relevant`) on candidate severe-finding sentences (`going_concern`, `default_or_recall`, `regulator_order`, `fraud`, `qualified_audit`) with surrounding $\pm 1$ sentence context.
- **Expected Compute Units:**
  - **`DEBUG = True` (T4 smoke run):** ~3 min (~0.08 Colab units). May run on unverified `ai` labels in `DEBUG=True` mode only to validate pipeline, checkpointing, ONNX export, and `metrics.json` schema.
  - **`DEBUG = False` (G4 full 3-seed run, `SEEDS = [13, 42, 2026]`, 4 epochs/seed):** ~20 min (**2.90 G4 units** at 8.71 units/hr). Requires Akshat's verified L2 labels (`human_verified_ai`).
- **Outputs on Drive (`/content/drive/MyDrive/finsight_p4/m3_event_clf/`):**
  - Epoch checkpoints: `epoch-<k>/` and `seed-<s>/epoch-<k>/`
  - Exported ONNX bundle: `onnx_fp32/{model.onnx, tokenizer.json, tokenizer_config.json, threshold.json, SHA256SUMS, WEIGHTS.md}`
  - Contract metrics: `metrics.json` (copy back to `eval_results/p4/argon/m3_event_clf/metrics.json`)

In [ ]:
# First cell of every ARGON notebook (from ml/handoff/colab/setup_cell.py)
import json
import os
import pathlib
import random
import subprocess
import sys

HF_REPO = "AkshatTm/finsight-gpu-jobs"  # private dataset repo (C-ADR-13)
HF_PREFIX = "phase4/handoff"  # folder in that repo; revision pinned in ml/handoff/README.md
HF_REVISION = "a285902d91a5557cf6d6d7cf23a9e266410d6c3d"
JOB = "m3_event_clf"
DEBUG = True  # True = tiny slice for T4 debug run; set False on G4 ONLY after DEBUG passes & L2 is verified
SEEDS = [13, 42, 2026]
RESUME = True  # continue from the newest checkpoint on Drive if one exists

ROOT = pathlib.Path("/content")
PACK = ROOT / "handoff"
DRIVE = pathlib.Path("/content/drive/MyDrive/finsight_p4")
CKPT_DIR = DRIVE / JOB


def mount_drive() -> None:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive not mounted:", exc)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)


def hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore

        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ["HF_TOKEN"]


def fetch_pack() -> pathlib.Path:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        revision=HF_REVISION,
        token=hf_token(),
        allow_patterns=[f"{HF_PREFIX}/*"],
        local_dir=str(ROOT / "_hf"),
    )
    src = ROOT / "_hf" / HF_PREFIX
    if PACK.exists():
        subprocess.run(["rm", "-rf", str(PACK)], check=True)
    subprocess.run(["cp", "-r", str(src), str(PACK)], check=True)
    return PACK


def verify_manifest() -> None:
    import hashlib

    manifest = json.loads((PACK / "data" / "MANIFEST.json").read_text(encoding="utf-8"))
    for f in manifest["files"]:
        h = hashlib.sha256((PACK / "data" / f["path"]).read_bytes()).hexdigest()
        assert h == f["sha256"], f"checksum mismatch: {f['path']}"


def install_deps() -> None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(PACK / "colab" / "requirements_colab.txt"),
        ],
        check=True,
    )


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def latest_checkpoint() -> pathlib.Path | None:
    ckpts = sorted(CKPT_DIR.glob("epoch-*"), key=lambda p: int(p.name.split("-")[1]))
    return ckpts[-1] if RESUME and ckpts else None


mount_drive()
fetch_pack()
verify_manifest()
install_deps()
sys.path.insert(0, str(PACK / "eval" / "vendor"))
print("pack ready:", PACK, "| DEBUG =", DEBUG, "| resume from:", latest_checkpoint())

In [ ]:
# Cell 2: Load L2 event_labels dataset, apply Akshat's verification CSV if present, and enforce 0 leakage
import csv
from collections import Counter

LABELS = ["actual_event", "hypothetical", "negated", "not_relevant"]
LABEL2ID = {lbl: idx for idx, lbl in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256


def resolve_event_labels_dir() -> pathlib.Path:
    candidates = [
        DRIVE / "event_labels",
        pathlib.Path("data/gold/event_labels"),
        pathlib.Path("../../data/gold/event_labels"),
        ROOT / "repo" / "data" / "gold" / "event_labels",
    ]
    for c in candidates:
        if (c / "train.jsonl").exists():
            return c
    # Shallow-clone p4/argon if running standalone on Colab
    repo_dir = ROOT / "repo"
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "-b",
                "p4/argon",
                "https://github.com/AkshatTm/FinSight-IPO-X-Ray-for-Indian-retail-investors.git",
                str(repo_dir),
            ],
            check=True,
        )
    return repo_dir / "data" / "gold" / "event_labels"


DATA_DIR = resolve_event_labels_dir()
print("Loading L2 event labels from:", DATA_DIR)


def load_jsonl(path: pathlib.Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_rows = load_jsonl(DATA_DIR / "train.jsonl")
dev_rows = load_jsonl(DATA_DIR / "dev.jsonl")
test_rows = load_jsonl(DATA_DIR / "test.jsonl")

# Apply Akshat's verification overrides from akshat_verification_gold.csv (Drive override takes priority)
csv_candidates = [
    DRIVE / "event_labels" / "akshat_verification_gold.csv",
    CKPT_DIR / "akshat_verification_gold.csv",
    DATA_DIR / "akshat_verification_gold.csv",
]
verified_map: dict[str, str] = {}
for csv_path in csv_candidates:
    if csv_path.exists():
        with csv_path.open("r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                v_lbl = (row.get("verified_label") or "").strip()
                band_val = (row.get("severity_band_1_to_4") or "").strip()
                src_val = (row.get("label_source") or "").strip()
                if v_lbl in LABEL2ID:
                    verified_map[row["id"]] = v_lbl
                elif band_val in {"1", "2", "3", "4"} or src_val in {"human", "human_verified_ai"}:
                    verified_map[row["id"]] = row["label"].strip()
        if verified_map:
            print(f"Applied {len(verified_map)} human-verified rows from {csv_path}")
            break

for split_rows in (train_rows, dev_rows, test_rows):
    for r in split_rows:
        if r["id"] in verified_map:
            r["label"] = verified_map[r["id"]]
            r["label_source"] = "human_verified_ai"

# Enforce zero excluded-IPO leakage on train & dev
excluded_obj = json.loads((PACK / "data" / "excluded_ipos.json").read_text(encoding="utf-8"))
excluded_ipos = set(excluded_obj["ipos"].keys())
for r in train_rows + dev_rows:
    assert r["ipo_id"] not in excluded_ipos, f"Excluded IPO leaked into train/dev: {r['ipo_id']}"

source_counts = Counter(r.get("label_source", "ai") for r in train_rows + dev_rows + test_rows)
print("Label source mix across all splits:", dict(source_counts))
if not DEBUG:
    assert source_counts["human_verified_ai"] + source_counts["human"] > 0, (
        "Final G4 run (DEBUG=False) requires Akshat's verified labels in akshat_verification_gold.csv!"
    )

if DEBUG:
    train_rows = train_rows[:32]
    dev_rows = dev_rows[:16]
    test_rows = test_rows[:16]
    SEEDS = [13]
    print(f"DEBUG mode active: train={len(train_rows)}, dev={len(dev_rows)}, test={len(test_rows)}, seeds={SEEDS}")

In [ ]:
# Cell 3: Train microsoft/deberta-v3-base across SEEDS with per-epoch Drive checkpointing & dev threshold tuning
import shutil
import time
import numpy as np
import torch
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

t0_wall = time.time()


def format_event_input(item: dict) -> str:
    check_id = str(item.get("check_id") or "unknown").strip()
    ctx_before = " ".join(str(item.get("context_before") or "").split())
    sentence = " ".join(str(item.get("sentence") or "").split())
    ctx_after = " ".join(str(item.get("context_after") or "").split())
    return f"[{check_id}] {ctx_before} [SEP] {sentence} [SEP] {ctx_after}".strip()


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class EventDataset(Dataset):
    def __init__(self, rows: list[dict]):
        self.rows = rows
        self.texts = [format_event_input(r) for r in rows]
        self.labels = [LABEL2ID[r["label"]] for r in rows]
        self.enc = tokenizer(
            self.texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
        )

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict:
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_ds = EventDataset(train_rows)
dev_ds = EventDataset(dev_rows)
test_ds = EventDataset(test_rows)

# Class weights (inverse frequency) so rare actual_event gets balanced gradient
train_label_counts = Counter(train_ds.labels)
total_train = len(train_ds)
class_weights = torch.tensor(
    [total_train / max(1, len(LABELS) * train_label_counts.get(i, 1)) for i in range(len(LABELS))],
    dtype=torch.float,
)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, len(LABELS)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class DriveEpochCheckpointCallback(TrainerCallback):
    """Saves `epoch-<k>` to `CKPT_DIR` and `CKPT_DIR/seed-<s>/epoch-<k>` at the end of every epoch."""

    def __init__(self, seed_dir: pathlib.Path):
        self.seed_dir = seed_dir

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_int = max(1, int(round(state.epoch or 1)))
        model = kwargs["model"]
        for target in (self.seed_dir / f"epoch-{epoch_int}", CKPT_DIR / f"epoch-{epoch_int}"):
            target.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(target))
            tokenizer.save_pretrained(str(target))
        return control


def softmax_np(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)


def apply_threshold_preds(probs: np.ndarray, ae_thresh: float) -> np.ndarray:
    preds = np.argmax(probs, axis=-1)
    ae_mask = probs[:, 0] >= ae_thresh
    preds[ae_mask] = 0
    non_ae_argmax = 1 + np.argmax(probs[:, 1:], axis=-1)
    preds[~ae_mask & (preds == 0)] = non_ae_argmax[~ae_mask & (preds == 0)]
    return preds


def tune_threshold_on_dev(dev_probs: np.ndarray, dev_y: np.ndarray) -> float:
    best_thresh = 0.50
    best_score = -1.0
    for t in np.linspace(0.20, 0.85, 66):
        preds = apply_threshold_preds(dev_probs, float(t))
        prec_ae = precision_score(dev_y == 0, preds == 0, zero_division=0)
        rec_ae = recall_score(dev_y == 0, preds == 0, zero_division=0)
        mac_f1 = f1_score(dev_y, preds, average="macro", zero_division=0)
        if prec_ae >= 0.90 and rec_ae >= 0.80 and mac_f1 > best_score:
            best_score = float(mac_f1)
            best_thresh = float(t)
    if best_score < 0:
        # Fallback if dev slice in DEBUG mode cannot hit both constraints simultaneously
        for t in np.linspace(0.25, 0.75, 51):
            preds = apply_threshold_preds(dev_probs, float(t))
            mac_f1 = f1_score(dev_y, preds, average="macro", zero_division=0)
            if mac_f1 > best_score:
                best_score = float(mac_f1)
                best_thresh = float(t)
    return round(best_thresh, 4)


num_epochs = 1 if DEBUG else 4
seed_metrics: list[dict] = []
best_dev_macro_f1 = -1.0
best_seed_dir: pathlib.Path | None = None
best_threshold = 0.50

for seed in SEEDS:
    set_seed(seed)
    seed_dir = CKPT_DIR / f"seed-{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    done_marker = seed_dir / "seed_result.json"

    if RESUME and done_marker.exists() and not DEBUG:
        cached = json.loads(done_marker.read_text(encoding="utf-8"))
        seed_metrics.append(cached)
        print(f"[Seed {seed}] Resumed finished result:", cached)
        if cached["dev_macro_f1"] > best_dev_macro_f1:
            best_dev_macro_f1 = cached["dev_macro_f1"]
            best_seed_dir = seed_dir / f"epoch-{num_epochs}"
            best_threshold = cached["threshold"]
        continue

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    training_args = TrainingArguments(
        output_dir=str(seed_dir / "hf_work"),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=8 if DEBUG else 16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        seed=seed,
        fp16=torch.cuda.is_available(),
        logging_steps=10,
        report_to="none",
    )
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        callbacks=[DriveEpochCheckpointCallback(seed_dir)],
    )
    trainer.train()

    dev_logits = trainer.predict(dev_ds).predictions
    dev_probs = softmax_np(dev_logits)
    dev_y = np.array(dev_ds.labels)
    tuned_t = tune_threshold_on_dev(dev_probs, dev_y)
    dev_preds = apply_threshold_preds(dev_probs, tuned_t)
    dev_mac_f1 = float(f1_score(dev_y, dev_preds, average="macro", zero_division=0))

    test_logits = trainer.predict(test_ds).predictions
    test_probs = softmax_np(test_logits)
    test_y = np.array(test_ds.labels)
    test_preds = apply_threshold_preds(test_probs, tuned_t)
    test_mac_f1 = float(f1_score(test_y, test_preds, average="macro", zero_division=0))
    test_prec_ae = float(precision_score(test_y == 0, test_preds == 0, zero_division=0))
    test_rec_ae = float(recall_score(test_y == 0, test_preds == 0, zero_division=0))

    res = {
        "seed": seed,
        "threshold": tuned_t,
        "dev_macro_f1": round(dev_mac_f1, 4),
        "test_macro_f1": round(test_mac_f1, 4),
        "test_precision_actual_event": round(test_prec_ae, 4),
        "test_recall_actual_event": round(test_rec_ae, 4),
    }
    done_marker.write_text(json.dumps(res, indent=2), encoding="utf-8")
    seed_metrics.append(res)
    print(f"[Seed {seed}] Done:", res)

    if dev_mac_f1 > best_dev_macro_f1:
        best_dev_macro_f1 = dev_mac_f1
        best_seed_dir = seed_dir / f"epoch-{num_epochs}"
        best_threshold = tuned_t

    shutil.rmtree(seed_dir / "hf_work", ignore_errors=True)

In [ ]:
# Cell 4: Export best seed to ONNX fp32, benchmark CPU latency (<= 50 ms/item), evaluate vs rules_v1, write metrics.json
import datetime
import hashlib
import onnxruntime as ort

assert best_seed_dir is not None and best_seed_dir.exists(), "No trained seed checkpoint found!"
onnx_dir = CKPT_DIR / "onnx_fp32"
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_path = onnx_dir / "model.onnx"

best_model = AutoModelForSequenceClassification.from_pretrained(str(best_seed_dir)).cpu().eval()
tokenizer.save_pretrained(str(onnx_dir))
(onnx_dir / "threshold.json").write_text(
    json.dumps({"actual_event_threshold": best_threshold, "labels": LABELS}, indent=2),
    encoding="utf-8",
)

dummy = tokenizer(
    ["[default_or_recall] Context before. [SEP] Sample sentence. [SEP] Context after."],
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
input_names = ["input_ids", "attention_mask"]
dummy_tuple = (dummy["input_ids"], dummy["attention_mask"])

torch.onnx.export(
    best_model,
    dummy_tuple,
    str(onnx_path),
    input_names=input_names,
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "logits": {0: "batch"},
    },
    opset_version=17,
    do_constant_folding=True,
)

# CPU latency benchmark via ONNX Runtime
sess_opt = ort.SessionOptions()
sess_opt.intra_op_num_threads = 4
ort_sess = ort.InferenceSession(str(onnx_path), sess_options=sess_opt, providers=["CPUExecutionProvider"])

t_lat0 = time.perf_counter()
for r in test_rows:
    enc_one = tokenizer(
        [format_event_input(r)],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="np",
    )
    _ = ort_sess.run(
        None,
        {
            "input_ids": enc_one["input_ids"].astype(np.int64),
            "attention_mask": enc_one["attention_mask"].astype(np.int64),
        },
    )
ms_per_item = ((time.perf_counter() - t_lat0) * 1000.0) / max(1, len(test_rows))
print(f"ONNX fp32 CPU latency: {ms_per_item:.2f} ms/item (limit <= 50 ms/item)")

# Compute rules_v1 baseline on the exact same test_rows
# In rules_v1, `rules_decision == "fired"` predicts `actual_event` (0), else `not_relevant` (3)
test_y_arr = np.array([LABEL2ID[r["label"]] for r in test_rows])
rules_preds_arr = np.array([0 if r.get("rules_decision") == "fired" else 3 for r in test_rows])
rules_macro_f1 = float(f1_score(test_y_arr, rules_preds_arr, average="macro", zero_division=0))
rules_prec_ae = float(precision_score(test_y_arr == 0, rules_preds_arr == 0, zero_division=0))
rules_rec_ae = float(recall_score(test_y_arr == 0, rules_preds_arr == 0, zero_division=0))

macro_f1s = [m["test_macro_f1"] for m in seed_metrics]
prec_aes = [m["test_precision_actual_event"] for m in seed_metrics]
rec_aes = [m["test_recall_actual_event"] for m in seed_metrics]

mean_macro_f1 = round(float(np.mean(macro_f1s)), 4)
std_macro_f1 = round(float(np.std(macro_f1s)), 4)
mean_prec_ae = round(float(np.mean(prec_aes)), 4)
mean_rec_ae = round(float(np.mean(rec_aes)), 4)

verdict = (
    "BEATS"
    if (mean_macro_f1 >= 0.82 and mean_prec_ae >= 0.90 and mean_rec_ae >= 0.80 and mean_macro_f1 > rules_macro_f1)
    else ("TIES" if abs(mean_macro_f1 - rules_macro_f1) <= 0.01 else "LOSES")
)

wall_minutes = round((time.time() - t0_wall) / 60.0, 2)
hw_name = "T4-DEBUG" if DEBUG else "G4"
units_rate = 1.60 if DEBUG else 8.71
units_used = round((wall_minutes / 60.0) * units_rate, 2)

metrics_payload = {
    "model_id": "m3_event_clf",
    "version": "v1-debug" if DEBUG else "v1",
    "task": "event classification",
    "split": "test (by IPO, excluded IPOs removed from train/dev)",
    "n_test": len(test_rows),
    "seeds": SEEDS,
    "primary_metric": {
        "name": "macro_f1",
        "mean": mean_macro_f1,
        "std": std_macro_f1,
    },
    "secondary": {
        "precision_actual_event": mean_prec_ae,
        "recall_actual_event": mean_rec_ae,
        "cpu_ms_per_item": round(ms_per_item, 2),
        "actual_event_threshold": best_threshold,
    },
    "baseline": {
        "name": "rules_v1",
        "primary": round(rules_macro_f1, 4),
        "precision_actual_event": round(rules_prec_ae, 4),
        "recall_actual_event": round(rules_rec_ae, 4),
        "source": "ml/handoff/eval/baselines.json",
    },
    "verdict": verdict,
    "label_source_mix": {
        "human": int(source_counts.get("human", 0)),
        "human_verified_ai": int(source_counts.get("human_verified_ai", 0)),
        "ai": int(source_counts.get("ai", 0)),
        "rule": int(source_counts.get("rule", 0)),
    },
    "compute": {
        "hardware": hw_name,
        "units_used": units_used,
        "wall_minutes": wall_minutes,
    },
    "date": datetime.date.today().isoformat(),
}

(CKPT_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

# Write SHA256SUMS and WEIGHTS.md inside onnx_dir
sha_lines = []
for p in sorted(onnx_dir.iterdir()):
    if p.is_file() and p.name not in {"SHA256SUMS", "WEIGHTS.md"}:
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        sha_lines.append(f"{h}  {p.name}")
(onnx_dir / "SHA256SUMS").write_text("\n".join(sha_lines) + "\n", encoding="utf-8")
(onnx_dir / "WEIGHTS.md").write_text(
    f"# M3 Event Classifier Weights (`m3_event_clf`)\n\n"
    f"- Drive path: `{onnx_dir}`\n"
    f"- Base model: `{MODEL_NAME}`\n"
    f"- Threshold (`actual_event`): `{best_threshold}`\n"
    f"- CPU latency: `{ms_per_item:.2f} ms/item`\n",
    encoding="utf-8",
)

print(json.dumps(metrics_payload, indent=2))